1. Feature Engineering & Son Hazırlıklar

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.feature_selection import VarianceThreshold

print("AŞAMA 2'de ayırdığımız setler yükleniyor...")

X_train = pd.read_parquet('../data/splits/X_train.parquet')
X_test = pd.read_parquet('../data/splits/X_test.parquet')

y_train = pd.read_parquet('../data/splits/y_train.parquet')

print(f"Başlangıçtaki özellik (sütun) sayısı: {X_train.shape[1]}")

# ---------------------------------------------------------
# 1. SIFIR VARYANSLI SÜTUNLARI DÜŞÜRME (Variance Threshold)
# ---------------------------------------------------------
print("\nSıfır varyanslı (gereksiz) özellikler temizleniyor...")
# Sadece X_train üzerinden öğrenme
selector = VarianceThreshold(threshold=0.0)
selector.fit(X_train)

# Tutulacak özellikleri al
features_to_keep = X_train.columns[selector.get_support()]
features_to_drop = [col for col in X_train.columns if col not in features_to_keep]

print(f"Silinen Sütun Sayısı: {len(features_to_drop)}")
if len(features_to_drop) > 0:
    print(f"Silinen Sütunlar: {features_to_drop}")

# Hem Train hem de Test setini bu güncel özelliklere göre filtrele
X_train = X_train[features_to_keep]
X_test = X_test[features_to_keep]

# ---------------------------------------------------------
# 2. CLASS IMBALANCE (SINIF DENGESİZLİĞİ) HESAPLAMASI
# ---------------------------------------------------------
# Label encoding yaptığımız için: Normal Trafik = 0, Saldırı = 1
neg_count = (y_train['Label'] == 0).sum()
pos_count = (y_train['Label'] == 1).sum()

# XGBoost scale_pos_weight formülü = (Negatif Sınıf Sayısı) / (Pozitif Sınıf Sayısı)
scale_pos_weight_value = neg_count / pos_count

print("\n--- XGBOOST DENGESİZLİK PARAMETRESİ ---")
print(f"Normal Trafik (0) Sayısı: {neg_count}")
print(f"Saldırı (1) Sayısı: {pos_count}")
print(f"Önerilen 'scale_pos_weight' değeri: {round(scale_pos_weight_value, 3)}")
print("Not: Bu değeri bir sonraki notebook'ta XGBoost modelini tanımlarken kullanacağız.")

# ---------------------------------------------------------
# 3. FİNAL LİSTELERİ VE SETLERİ KAYDETME
# ---------------------------------------------------------
final_features = list(X_train.columns)

# Kullanılan özellikleri JSON olarak kaydet
with open('../data/splits/final_features.json', 'w') as f:
    json.dump(final_features, f)

# Güncellenmiş X setlerini üzerine yazıp kaydetme
X_train.to_parquet('../data/splits/X_train.parquet', index=False)
X_test.to_parquet('../data/splits/X_test.parquet', index=False)

print("\nHarika! Tüm gereksiz sütunlar temizlendi, özellik listesi ve yeni veri setleri kaydedildi.")

AŞAMA 2'de ayırdığımız setler yükleniyor...
Başlangıçtaki özellik (sütun) sayısı: 77

Sıfır varyanslı (gereksiz) özellikler temizleniyor...
Silinen Sütun Sayısı: 8
Silinen Sütunlar: ['Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Byts/b Avg', 'Fwd Pkts/b Avg', 'Fwd Blk Rate Avg', 'Bwd Byts/b Avg', 'Bwd Pkts/b Avg', 'Bwd Blk Rate Avg']

--- XGBOOST DENGESİZLİK PARAMETRESİ ---
Normal Trafik (0) Sayısı: 7749281
Saldırı (1) Sayısı: 853555
Önerilen 'scale_pos_weight' değeri: 9.079
Not: Bu değeri bir sonraki notebook'ta XGBoost modelini tanımlarken kullanacağız.

Harika! Tüm gereksiz sütunlar temizlendi, özellik listesi ve yeni veri setleri kaydedildi.


scale_pos_weight değeri 9.079 çıktı. Bu siber güvenlik dünyasında çok gerçekçi bir tablodur; yani ağdaki normal trafik, saldırılardan yaklaşık 9 kat daha fazla. Normalde yapay zeka bu çoğunluğa aldanıp saldırıları gözden kaçırabilirdi (Accuracy paradoksu).